In [3]:
#README


In [10]:
#IMPORTS

import yfinance as yf
import pandas as pd
import numpy as np
from rich.jupyter import display


In [5]:
#INPUTS

target = yf.Ticker("MSFT")

benchmark_2 = yf.Ticker("URTH")
benchmark = "URTH"
start_date = "2020-12-31"
end_date = "2025-12-31"
save_data = True
period = "5y"
interval = "1wk"

peer_group = ["ORCL", "PLTR", "PANW", "CRWD", "FTNT"]


In [6]:
#FUNCTIONS
def get_data(ticker: str, start_date: str, end_date: str, interval: str, save_data: bool):
    data = yf.download(ticker, start_date, end_date, interval )
    if save_data:
        path = f"./data/{ticker}_{start_date}_-_{end_date}.csv"
        data.to_csv(path, index=False)
    return data["Close"]


In [35]:
#DATA

#BENCHMARK

benchmark_data = get_data(benchmark, start_date, end_date, interval, save_data)
benchmark_data = benchmark_data.dropna()
log_ret = np.log(benchmark_data / benchmark_data.shift(1))
benchmark_data.insert(1, f"{benchmark}log_ret /BENCHMARK/", log_ret)
benchmark_data = benchmark_data.dropna()

# PEER_GROUP

peer_results = {}
for peer in peer_group:
    peer_results[peer] = get_data(peer, start_date, end_date, interval, save_data)

peer_df = pd.concat(peer_results, axis=1)
peer_df.columns = peer_df.columns.get_level_values(0)
peer_df.dropna()

for peer in peer_group:
    col_idx = peer_df.columns.get_loc(peer)
    log_ret = np.log(peer_df[peer] / peer_df[peer].shift(1))
    peer_df.insert(col_idx + 1, f"{peer}_log_ret", log_ret)

peer_df = peer_df.dropna()

#MERGE & SAVE

combined_df = benchmark_data.join(peer_df)
combined_df.to_csv("./data/combined_df.csv", index=True)


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [8]:

#target_info
ticker = target.ticker
comp_name = target.info["shortName"]
country = target.info["country"]
sector = target.info["sector"]
industry = target.info["industry"]
industry_key = target.info["industryKey"]

benchmark_ticker = benchmark.ticker
benchmark_name = benchmark.info["shortName"]


print(f"Benchmark: {benchmark_name}\n")
print(f"Target: {comp_name}\nTicker: {ticker}\nCountry: {country}\nSector: {sector}\nIndustry: {industry}\n")
# industry_info
ind = yf.Industry(industry_key)
display(ind.top_companies)

#target_close
# data = target.history(period="1y", interval="1wk")
# display(data[["Close"]])



AttributeError: 'str' object has no attribute 'ticker'